# Marketing Source and Campaign Effectiveness

This notebook evaluates acquisition performance across marketing sources and campaigns.

### Business questions
1. Which sources generate the most leads and buyers?
2. Which sources achieve the strongest lead-to-buyer conversion?
3. How does lead quality differ across acquisition sources?
4. Which paid sources have high acquisition cost?
5. Which campaigns provide the best balance between scale and conversion?

> Source-level CAC is interpreted only where tracked marketing spend and buyer counts are available.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from helpers import (
    colors,
    descriptive_stats,
    cat_stats,
    date_stats,
    plot_distributions,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


## Load Processed Data

In [ ]:
deals = pd.read_pickle(PROCESSED_DIR / 'deals_clean.pkl')
spend = pd.read_pickle(PROCESSED_DIR / 'spend_clean.pkl')
buyers = pd.read_pickle(PROCESSED_DIR / 'buyers.pkl')

## 1. Source Effectiveness

### 1.1 Lead Definition and Preparation

In [ ]:
leads = deals.copy()

In [ ]:
# A lead is defined as a unique contact associated with at least one CRM deal.

total_leads = leads['contact_name'].nunique()     
total_buyers = len(buyers['contact_name'])

print(f"Unique leads: {total_leads:,}")
print(f"Buyers: {total_buyers:,}")
print(f"Lead-to-buyer conversion: {total_buyers/total_leads:.1%}")

### 1.2 Source-Level Performance

In [ ]:
source_agg = pd.concat([
    leads.groupby('source')['contact_name'].nunique().rename('leads_count'),
    buyers.groupby('source')['contact_name'].nunique().rename('buyers_count'),
], axis=1).fillna(0).reset_index()

source_agg['conv_rate'] = (source_agg['buyers_count'] / source_agg['leads_count'] * 100).round(1)
source_agg = source_agg.sort_values('leads_count', ascending=False).reset_index(drop=True)
print(source_agg)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Mean values used as quadrant reference lines
avg_leads = source_agg['leads_count'].mean()
avg_conv  = source_agg['conv_rate'].mean()

ax.scatter(source_agg['leads_count'], source_agg['conv_rate'], s=source_agg['buyers_count'] * 5, alpha=0.7, color=colors['accent'])
# Bubble size represents buyers; alpha controls transparency

# Quadrant reference lines
ax.axhline(y=avg_conv,  color=colors['grid'], linestyle='--', alpha=0.8)
ax.axvline(x=avg_leads, color=colors['grid'], linestyle='--', alpha=0.8)

# Quadrant labels

# Maximum conversion value used to position quadrant labels
ymax = source_agg['conv_rate'].max()

# Position labels away from plotted points
ax.text(avg_leads * 1.6, ymax * 0.95, 'High L / High C', fontsize=9, color='green', fontweight='bold')
ax.text(avg_leads * 0.1, ymax * 0.95, 'Low L / High C',    fontsize=9, color='steelblue', fontweight='bold')
ax.text(avg_leads * 1.6, avg_conv * 0.15, 'High L / Low C',fontsize=9, color='orange', fontweight='bold')
ax.text(avg_leads * 0.1, avg_conv * 0.15, 'Low L / Low C', fontsize=9, color='red', fontweight='bold')

# Label each acquisition source
for source, x, y in zip(source_agg['source'], source_agg['leads_count'], source_agg['conv_rate']):
    ax.annotate(source, (x, y), textcoords='offset points', xytext=(5, 5),
                fontsize=9, color=colors['text'])
# annotate() adds a label to a plotted point

ax.set_xlabel('L (Leads)')
ax.set_ylabel('C (Conversion %)')
ax.set_title('Sources: leads vs conversion (size = buyers)')
plt.tight_layout()
plt.show()

In [ ]:
# Lead-quality distribution within each source (%)
quality_by_source = (pd.crosstab(leads['source'], leads['quality'], normalize='index') * 100).round(1)
# normalize='index' calculates row percentages

# Keep the three target quality categories
quality_order = ['A - High', 'B - Medium', 'C - Low']
quality_by_source = quality_by_source[[c for c in quality_order if c in quality_by_source.columns]]
# Keep only quality categories present in the dataset

# Plot
fig, ax = plt.subplots(figsize=(20, 10))

quality_by_source.plot(kind='barh', stacked=True, color=[colors['accent'], '#10B981', '#F59E0B'], ax=ax)

ax.set_title('Lead Quality Distribution by Source', fontsize=14, fontweight='bold')
ax.set_xlabel('Share (%)')
ax.set_ylabel('')

# Legend
ax.legend(title='Quality', loc='lower right')

# Add percentage labels to stacked bars
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        label_type='center',
        fontsize=15,
        color='white'
    )

plt.tight_layout()
plt.show()

In [ ]:
# Compare source performance with tracked marketing spend

In [ ]:
spend_by_source_df = spend.groupby('source')['spend'].sum().reset_index().rename(columns={'spend': 'total_spend'})
source_agg = source_agg.merge(spend_by_source_df, on='source', how='left').fillna(0)
source_agg['cac'] = (source_agg['total_spend'] / source_agg['buyers_count']).replace([float('inf')], 0).round(0)

In [ ]:
spend_by_source = (spend.groupby('source')['spend'].sum().sort_values(ascending=False))

plt.figure(figsize=(10, 5))

sns.barplot(x=spend_by_source.index, y=spend_by_source.values, color=colors['accent'])

plt.title('Marketing Spend by Source')
plt.xlabel('')
plt.ylabel('Spend')

plt.xticks(rotation=45, ha='right')

# Add spend labels above bars
for i, value in enumerate(spend_by_source.values):
    plt.text(i, value, f'{value:,.0f}', ha='center', va='bottom') # place the label above the bar

plt.tight_layout()
plt.show()

In [ ]:
source_agg['cac'] = np.where(
    source_agg['buyers_count'] > 0,
    source_agg['total_spend'] / source_agg['buyers_count'],
    np.nan
)
source_agg['cac'] = source_agg['cac'].round(0)

print(
    source_agg[
        ['source', 'leads_count', 'buyers_count', 'conv_rate', 'total_spend', 'cac']
    ].reset_index(drop=True)
)

### Source-Level Findings

- **Organic** has the highest lead-to-buyer conversion at **9.8%** and no tracked acquisition spend.
- **Webinar** combines a high conversion rate (**8.8%**) with relatively low CAC.
- **SMM** shows a lower CAC (**€84**) with a conversion rate of **5.4%**.
- **Google Ads** and **Bloggers** have the highest CAC values (about **€346** and **€345**) despite only moderate conversion.
- **Facebook Ads** provides the largest lead volume (**4,423 leads**) but at a higher CAC than several other paid sources.
- CRM and Partnership have low conversion, but they represent different acquisition flows and should not be interpreted in exactly the same way as paid advertising channels.

**Business implication:** higher marketing spend does not automatically translate into higher conversion efficiency.

## 2. Campaign-Level Performance

In [ ]:
camp_agg = pd.concat([
    leads.groupby('campaign')['contact_name'].nunique().rename('leads_count'),
    buyers.groupby('campaign')['contact_name'].nunique().rename('buyers_count'),
], axis=1).fillna(0).astype(int).reset_index()

camp_agg['conv_rate'] = (camp_agg['buyers_count'] / camp_agg['leads_count'] * 100).round(1)

camp_agg = camp_agg.sort_values('leads_count', ascending=False).reset_index(drop=True)
print(camp_agg.head(20))

In [ ]:
top_camp = (camp_agg.sort_values('leads_count', ascending=False).head(10))

fig, ax1 = plt.subplots(figsize=(12, 6))

# Leads
ax1.bar(top_camp['campaign'], top_camp['leads_count'], color='#E2E8F0',label='Leads')

# Buyers
ax1.bar(top_camp['campaign'], top_camp['buyers_count'], color=colors['accent'], label='Buyers')

ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=50)

# Secondary axis
ax2 = ax1.twinx()

ax2.plot(top_camp['campaign'], top_camp['conv_rate'], color='#10B981', marker='o', linewidth=2, label='Conversion %')

ax2.set_ylabel('Conversion (%)', color='#10B981')
ax2.tick_params(axis='y', labelcolor='#10B981')

# Combined legend
# Combine handles and labels from both axes
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()  # Conversion line

ax1.legend(h1 + h2, l1 + l2)

plt.title('Top Campaigns: Leads, Buyers and Conversion')
plt.tight_layout()
plt.show()

### Campaign-Level Findings

- **brand_search_eng_DE** has the highest conversion among the highlighted campaigns (**9.4%**) but operates at small scale with 159 leads.
- **performancemax_digitalmarkt_ru_DE** generates the largest lead volume (**2,471 leads**) with a **4.4%** conversion rate, providing a stronger balance between scale and conversion.
- **02.07.23wide_DE** reaches **5.5%** conversion among the larger-volume campaigns.
- **webinar1906** and **performancemax_eng_DE** show very low conversion and warrant further review.

**Business implication:** campaign decisions should consider both scale and conversion rather than ranking campaigns by a single metric.